Выбор большой языковой модели

In [1]:
from llamba.chatmodels.ollama import OllamaModel
chatbot = OllamaModel(url="http://127.0.0.1:11434/", 
                      endpoint="api/generate", 
                      model="deepseek-r1", 
                      num_threads=1, 
                      check_connection_timeout=15,  
                      request_timeout=60,
                      think=False)
connection = chatbot.check_connection()
print(connection)

True


Подготовка данных

In [2]:
import torch
from torch import nn
import pandas as pd
import numpy as np

from llamba_library.bioage_model import BioAgeModel
from llamba_library.functions import get_shap_dict
from txai_omics_3.models.tabular.widedeep.ft_transformer \
    import WDFTTransformerModel, FN_SHAP, FN_CHECKPOINT, TRAIN_DATA_PATH


#### Данные
feats = [
    'CXCL9',
    'CCL22',
    'IL6',
    'PDGFB',
    'CD40LG',
    'IL27',
    'VEGFA',
    'CSF1',
    'PDGFA',
    'CXCL10'
]

dataset_path = TRAIN_DATA_PATH
train_data = pd.read_excel(TRAIN_DATA_PATH)
train_data = train_data[train_data['Split'] == 'trn_val'].copy()
train_data = train_data.loc[:, feats + ['Age']]

bioage_data_dict = {'CXCL9': 2599.629474, 
           'CCL22': 820.306524, 
           'IL6': 0.846377, 
           'PDGFB': 13400.666359, 
           'CD40LG': 1853.847406, 
           'IL27': 1128.886982,
           'VEGFA': 153.574220,
           'CSF1': 239.627236,
           'PDGFA': 1005.844290,
           'CXCL10': 228.229829,
           'Age': 90.454972 }

bioage_data = pd.DataFrame(bioage_data_dict, index=[0])

#### Модель
    
fn_model = FN_CHECKPOINT
model = WDFTTransformerModel.load_from_checkpoint(fn_model)
bioage_model = BioAgeModel(model=model)

from llamba.util.bioage_module import BioAgeModule

bioage_module = BioAgeModule(bioage_model, feats=list(bioage_data_dict.keys()))

def predict_func(x):
    batch = {
        'all': torch.from_numpy(np.float32(x)),
        'continuous': torch.from_numpy(np.float32(x))
    }
    return model(batch).cpu().detach().numpy()

top_n = 3 # количество признаков с наибольшим вкладом#


#### Оценка риска смертности

##### Глиома

from llamba.util.mortality_predictor import MortalityPredictor
from xai_glioma_diagnostics.mortality_predictor import mortality_predictor

glioma_data_dict = {
            'TERT': 0, 
            'NOX4': 17, 
            'MMP9': 24, 
            'TRIM67': 175, 
            'ZDHHC18': 1235, 
            'HDAC1': 1538,
            'TUBB6': 641,
            'ADM': 339,
            'NOG': 357,
            'CHEK2': 147,
            'KCNJ11': 315,
            'KCNIP2': 2790,
            'VEGFA': 153.574220}
glioma_data = pd.DataFrame(glioma_data_dict, index=[0])

glioma_mortality_predictor_model = mortality_predictor(filepath_info="H:/Lobachevsky/XAI-glioma-diagnostics/models/sp_info.pkl", filepath_net="H:/Lobachevsky/XAI-glioma-diagnostics/models/sp.pt")
glioma_mortality_predictor = MortalityPredictor(model=glioma_mortality_predictor_model, disease_name='glioma', feats=list(glioma_data_dict.keys()))

##### Сахарный диабет 2-го типа

from xai_t2dm_mortality_prediction.t2dm_model import t2dm_model

t2d_data_dict = {
            'Age': 90.454972,
            'Number of complications': 0,
            'NT-proBNP': 31,
            'Triglycerides': 137,
            'Creatinine': 0.97,
            'hs-CRP': 2.67,
            'RDW-SD': 41.6,
            'Apolipoprotein A1': 149,
            'N-glycan NA3F': 1.88,
            'Disease duration': 5
}
t2d_data = pd.DataFrame(t2d_data_dict, index = [0])

t2d_mortality_predictor_model = t2dm_model(filepath_model_params="H:/Lobachevsky/XAI-T2DM-mortality-prediction/model/model.pkl")
t2d_mortality_predictor = MortalityPredictor(t2d_mortality_predictor_model, disease_name='type 2 diabetes', feats=list(t2d_data_dict.keys()))

#### Формирование большого датафрейма со всеми данными о пациенте

merged = {**bioage_data_dict, **glioma_data_dict, **t2d_data_dict}
big_df = pd.DataFrame([merged])
big_df

h:\Lobachevsky\llamba\venv\Lib\site-packages\pytorch_lightning\utilities\migration\utils.py:55: The loaded checkpoint was produced with Lightning v2.4.0, which is newer than your current Lightning version: v2.1.4
h:\Lobachevsky\llamba\venv\Lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Metric `SpearmanCorrcoef` will save all targets and predictions in the buffer. For large datasets, this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)  # noqa: B028


,CXCL9,CCL22,IL6,PDGFB,CD40LG,IL27,VEGFA,CSF1,PDGFA,CXCL10,...,KCNIP2,Number of complications,NT-proBNP,Triglycerides,Creatinine,hs-CRP,RDW-SD,Apolipoprotein A1,N-glycan NA3F,Disease duration
0,2599.629474,820.306524,0.846377,13400.666359,1853.847406,1128.886982,153.57422,239.627236,1005.84429,228.229829,...,2790,0,31,137,0.97,2.67,41.6,149,1.88,5


Инициализация коннектора

In [3]:
from llamba.connector import LlambaConnector

connector = LlambaConnector(language="ru", chat_model=chatbot)
connector.add_module(bioage_module)
connector.add_module(glioma_mortality_predictor)
connector.add_module(t2d_mortality_predictor)

Инициализация векторной базы данных

In [4]:
from llamba.util.document_repo import DocumentRepoQdrant

document_repo = DocumentRepoQdrant(url="127.0.0.1", 
                                   port="6060", 
                                   collection_name="ageing_docs",
                                   payload_return="data",
                                   max_results=5)
connector.specify_db(document_repo)

H:\Lobachevsky\llamba\llamba\util\document_repo.py:5: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  self.qdrant_client = QdrantClient(host=url, port=port)


Указание калькулятора биологических часов

In [5]:
connector.specify_clock_name(name="Immunoage", doi="10.1007/s11357-022-00540-4")

Указание производственных условий

In [6]:
connector.specify_conditions(condition="mining industry", duration="15 years")

Передача экспертной системе данных 

In [7]:
res = connector.analyze(data=big_df,
                        top_n=3,
                        analyze_feats=True,
                        analyze_risks=True, 
                        train_data=train_data, 
                        predict_func=predict_func)
print(res['analysis'])

### Отчет по анализам

#### 1) Биологический возраст и ускорение старения
Биологический возраст: 79 лет  
Ускорение старения: -11 лет (стареете медленнее нормы для вашего возраста)

#### 2) Наиболее значимые параметры и их значения
| Признак          | Значение       | Ускорение старения |
|-------------------|---------------|---------------------|
| IL6              | 0.846377      | -4.859              |
| CD40LG           | 1853.847406   | 7.278               |
| CXCL9            | 2599.629474   | -10.134             |

#### 3) Описание параметров и отклонение от нормы
- **IL6 (интерлейкин-6)**:  
  - **Описание**: Цитокин, ключевой в иммунных процессах и регуляции воспаления.  
  - **Отклонение**: Значение 0.846377 ниже порога нормы, что указывает на замедление воспалительного процесса. Это может свидетельствовать о замедленном старении или сниженной воспалительной нагрузке.  

- **CD40LG**:  
  - **Описание**: Молекула, экспрессируемая T-хелпером класса 1 (Th1), участвующая в регу

In [8]:
recommendations = connector.produce_recommendations()
print(recommendations)

На основании предоставленных данных и учета профессиональной истории (работа в горнодобывающей промышленности 15 лет), рекомендации для нормализации показателей и снижения риска заболеваний включают:

1. **Контроль воспалительных процессов**: 
   - Мониторинг уровня IL-6 (оптимальный диапазон: 1-7 пг/мл). Низкий уровень (0.846) может способствовать хроническому воспалению. Рекомендуется прием противовоспалительных препаратов (например, ацетилсалицилата) в рекомендуемых дозах.
   - Снижение риска ТБ и ЗППП через регулярные тесты на инфекции и вакцинацию.

2. **Иммуномодуляция**:
   - Снижение высокого уровня CD40LG (1853.847) может нарушать баланс Th1/Th2. Рекомендуется прием иммуносупрессантов (например, метотрексата) для нормализации иммунного ответа.
   - Низкий уровень CXCL9 (2599.629) указывает на снижение клеточного иммунитета. Введение терапии с хемокинами или стимуляторами иммунитета.

3. **Профилактика онкологических заболеваний**:
   - Регулярные скрининги (МРТ, биопсия) для р